# Differential Testing
Identifying functional differences between two code candidates by generating different inputs

In [336]:
from common.coevolution import lcb_dataset
problems = lcb_dataset.load_code_generation_dataset(
    release_version="release_v6",
    start_date="2025-03-01",
    end_date="2025-05-10",
    difficulty=lcb_dataset.Difficulty.HARD,
)

2025-12-19 21:14:32.826 | INFO     | common.coevolution.lcb_dataset:load_code_generation_dataset:127 - Using 'test' split of the dataset.
2025-12-19 21:14:55.681 | INFO     | common.coevolution.lcb_dataset:load_code_generation_dataset:141 - Filtered problems by difficulty: hard
2025-12-19 21:14:55.851 | INFO     | common.coevolution.lcb_dataset:load_code_generation_dataset:151 - Loaded 37 problems


In [337]:
for problem in problems:
    print(problem.public_test_cases[0].testtype)

TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.FUNCTIONAL
TestType.FUNCTIONAL
TestType.FUNCTIONAL
TestType.FUNCTIONAL
TestType.FUNCTIONAL
TestType.FUNCTIONAL
TestType.FUNCTIONAL
TestType.FUNCTIONAL
TestType.FUNCTIONAL


In [341]:
from common.coevolution.lcb_dataset import TestType 
problem_id = 'abc395_f'

problem = next((p for p in problems if p.question_id == problem_id), None)
if problem is None:
    raise ValueError(f"Problem with ID {problem_id} not found.")


# problem = [p for p in problems if p.public_test_cases[0].testtype == TestType.STDIN][1]

In [342]:
print(problem.starter_code)


class Solution:
    def sol(self, input_str: str) -> str:
        # Input is provided as a single string.



In [343]:
print(problem.question_content)

Takahashi has 2N teeth: N upper teeth and N lower teeth.
The length of the i-th upper tooth from the left (1 \leq i \leq N) is U _ i, and the length of the i-th lower tooth from the left (1 \leq i \leq N) is D _ i.
His teeth are said to “fit together well” if both of the following conditions are satisfied:

- There exists an integer H such that U _ i + D _ i = H for every integer i with 1 \leq i \leq N.
- \lvert U _ i - U _ {i+1} \rvert \leq X for every integer i with 1 \leq i < N.

He can perform the following operation any number of times:

- Pay 1 yen to use a tooth-grinding machine, choose exactly one tooth whose length is positive, and reduce its length by 1.

No other method may be used to change the lengths of the teeth.
Find the minimum total amount of money he needs to pay to make his teeth fit together well.

Input

The input is given from Standard Input in the following format:
N X
U _ 1 D _ 1
U _ 2 D _ 2
\vdots
U _ N D _ N

Output

Print the answer.

Constraints


- 2 \leq 

In [344]:
problem.public_test_cases

[LCBTest(input='4 3\n3 1\n4 1\n5 9\n2 6', output='15', testtype=<TestType.STDIN: 'stdin'>),
 LCBTest(input='4 1000000000\n3 3\n3 3\n3 3\n3 3', output='0', testtype=<TestType.STDIN: 'stdin'>),
 LCBTest(input='4 1\n1000000000 1000000000\n1000000000 1000000000\n1000000000 1000000000\n1 1', output='5999999994', testtype=<TestType.STDIN: 'stdin'>),
 LCBTest(input='15 128\n748 169\n586 329\n972 529\n432 519\n408 587\n138 249\n656 114\n632 299\n984 755\n404 772\n155 506\n832 854\n353 465\n387 374\n567 385', output='9460', testtype=<TestType.STDIN: 'stdin'>)]

# Candidate Code Solutions

In [346]:
C0_snippet = """
class Solution:
    def sol(self, input_str: str) -> str:
        \"\"\"
        Approach 3: Dynamic programming enforcing difference constraint with range of possible upper values
        Explanation:
        - For each position i, possible final upper tooth values lie in [0, U_i] and also equal H - D_i.
        - We treat H as parameter; iterate H over unique S_i (at most N).
        - For fixed H, target value at i is T_i = H - D_i must be in [0, U_i].
        - Now enforce adjacent difference <= X: if T_i feasible, we compute minimal cost to achieve adjusted values by DP: keep single value A_i after clamping using greedy since cost is linear in reduction. Implementation uses two-pass clamping (same as earlier) but coded with clear checks.
        - This variant focuses on robust per-H DP style logic.
        Complexity: O(N * M) where M number of candidate H (<=N).
        \"\"\"
        data = list(map(int, input_str.strip().split()))
        it = iter(data)
        N = next(it); X = next(it)
        U = []; D = []; S = []
        for _ in range(N):
            u = next(it); d = next(it)
            U.append(u); D.append(d); S.append(u + d)
        candidates = sorted(set(S) | {0})
        minS = min(S)
        candidates = [h for h in candidates if h <= minS]
        INF = 10**30
        best = INF
        for H in candidates:
            T = [H - d for d in D]
            feasible = True
            for i in range(N):
                if T[i] < 0 or T[i] > U[i]:
                    feasible = False
                    break
            if not feasible:
                continue
            A = T[:]
            for i in range(1, N):
                if A[i] > A[i-1] + X:
                    A[i] = A[i-1] + X
            for i in range(N-2, -1, -1):
                if A[i] > A[i+1] + X:
                    A[i] = A[i+1] + X
            if any(a < 0 for a in A):
                continue
            cost = 0
            for i in range(N):
                cost += (U[i] - A[i]) + (D[i] - (H - A[i]))
            if cost < best:
                best = cost
        return str(best)
"""


C1_snippet = """
class Solution:
    def sol(self, input_str: str) -> str:
        \"\"\"
        Approach 2: Binary search on H with greedy projection (O(N log W))
        Explanation:
        - Cost(H) is monotone nonincreasing as H decreases? Not strictly, but feasible set grows as H decreases.
        - We binary-search over H in [0, min(S_i)] to find minimal achievable total cost by evaluating cost(H).
        - For fixed H, compute minimal cost by projecting target T_i = H - D_i, clamping to [0, U_i], then enforcing adjacent diff constraint by two passes as min-heap style but here simple passes suffice.
        - Return minimal cost encountered during binary search neighborhood (we sample around found boundary).
        Complexity: Each cost computation is O(N). Binary search over 0..1e9 uses ~60 steps.
        \"\"\"
        data = list(map(int, input_str.strip().split()))
        it = iter(data)
        N = next(it); X = next(it)
        U = []; D = []; S = []
        for _ in range(N):
            u = next(it); d = next(it)
            U.append(u); D.append(d); S.append(u + d)
        lo = 0
        hi = min(S)
        INF = 10**30
        def eval_cost(H):
            T = [H - d for d in D]
            for t in T:
                if t < 0:
                    return INF
            A = [min(t, U[i]) for i, t in enumerate(T)]
            # enforce differences by reducing where necessary
            for i in range(1, N):
                if A[i] > A[i-1] + X:
                    A[i] = A[i-1] + X
            for i in range(N-2, -1, -1):
                if A[i] > A[i+1] + X:
                    A[i] = A[i+1] + X
            for a in A:
                if a < 0:
                    return INF
            total = 0
            for i in range(N):
                total += (U[i] - A[i]) + (D[i] - (H - A[i]))
            return total
        # binary search to find region of minimal cost; since cost may be nonconvex, sample around
        best = INF
        # perform binary search for minimal cost by checking mid and neighbors
        for _ in range(60):
            mid = (lo + hi) // 2
            c = eval_cost(mid)
            best = min(best, c)
            # compare mid and mid+1 to move direction
            c1 = eval_cost(mid+1) if mid+1 <= hi else INF
            if c1 < c:
                lo = mid + 1
            else:
                hi = mid - 1
        # sample a small neighborhood around final lo..hi
        for h in range(max(0, lo-5), min(min(S), lo+5)+1):
            best = min(best, eval_cost(h))
        return str(best)
"""

In [347]:
# Creating code individuals and populations
from common.coevolution.core.individual import CodeIndividual
from common.coevolution.core.population import CodePopulation
from common.coevolution.core.interfaces import Operations


c0_ind = CodeIndividual(
    snippet=C0_snippet,
    probability=0.75,
    creation_op=Operations.INITIAL,
    generation_born=0,
    parent_ids=[],
)

c1_ind = CodeIndividual(
    snippet=C1_snippet,
    probability=0.75,
    creation_op=Operations.INITIAL,
    generation_born=0,
    parent_ids=[],
)


code_pop = CodePopulation(individuals=[c0_ind, c1_ind], generation=0)

2025-12-19 22:02:14.085 | DEBUG    | common.coevolution.core.individual:__init__:42 - Created new <CodeIndividual id=C14 gen=0 op=initial prob=0.75>
2025-12-19 22:02:14.086 | DEBUG    | common.coevolution.core.individual:__init__:42 - Created new <CodeIndividual id=C15 gen=0 op=initial prob=0.75>
2025-12-19 22:02:14.087 | DEBUG    | common.coevolution.core.interfaces:__init__:479 - Initialized CodePopulation with 2 individuals, gen 0


In [348]:
from common.code_preprocessing.composition import compose_lcb_output_script
from common.coevolution.lcb_dataset import LCBTest
import json
import re


def get_inputdata_dict_from_functional_test_case(pub_test_case: LCBTest, starter_code: str) -> dict:
    if not isinstance(pub_test_case, LCBTest):
        raise TypeError("pub_test_case must be an instance of LCBTest")
    
    if pub_test_case.testtype != TestType.FUNCTIONAL:
        raise ValueError("pub_test_case must be of FUNCTIONAL test type")
    
    # Regex to find the first method defined with 'self', and extract its arguments
    func_match = re.search(r'def\s+\w+\s*\(self\s*,\s*([^\)]*)\)', starter_code)
    if not func_match:
        raise ValueError("Could not find a method with 'self' in the starter code.")
    func_args = func_match.group(1).strip().split(',')
    func_args = [arg.split(":")[0].strip() for arg in func_args if arg.strip()]

    input_args_list = [eval(arg) for arg in pub_test_case.input.split("\n")]
    if len(input_args_list) != len(func_args):
        raise ValueError("Number of input arguments does not match number of function parameters.")
    
    input_data_dict = dict(zip(func_args, input_args_list))
    return input_data_dict
    
def compose_output_script_from_public_test_case(code_snippet: str, pub_test_case: LCBTest, starter_code: str) -> str:
    if not isinstance(pub_test_case, LCBTest):
        raise TypeError("pub_test_case must be an instance of LCBTest")
    
    if pub_test_case.testtype == TestType.STDIN:
        input_data = json.dumps({'inputdata': {'input_str': pub_test_case.input}})

    elif pub_test_case.testtype == TestType.FUNCTIONAL:
        input_data_dict = get_inputdata_dict_from_functional_test_case(pub_test_case, starter_code)
        
        input_data = json.dumps({'inputdata': input_data_dict})

    composed_script = compose_lcb_output_script(programmer_code=code_snippet, input_data=input_data)
    return composed_script

In [349]:
import importlib
from common.sandbox import create_safe_test_environment

sandbox = create_safe_test_environment()

functionally_equivalent = True
public_test_input_outputs = []
for test in problem.public_test_cases:
    print(f"Executing test with input: {test.input}")
    c0_script = compose_output_script_from_public_test_case(c0_ind.snippet, test, starter_code=problem.starter_code)
    c0_result = sandbox.execute_code(c0_script)

    c1_script = compose_output_script_from_public_test_case(c1_ind.snippet, test, starter_code=problem.starter_code)
    c1_result = sandbox.execute_code(c1_script)

    print(f"C0 Output: {c0_result.output.strip()}, C1 Output: {c1_result.output.strip()}, Expected: {test.output.strip()}")


    if c0_result.output.strip() != c1_result.output.strip():
        functionally_equivalent = False
        print(f"The two code snippets are NOT functionally equivalent on problem {problem.question_id}.")
        break

    public_test_input_outputs.append({'inputdata':test.input if test.testtype == TestType.STDIN else get_inputdata_dict_from_functional_test_case(test, problem.starter_code), 'output': c0_result.output.strip()})

if functionally_equivalent:
    print(f"The two code snippets are functionally equivalent on all public test cases of problem {problem.question_id}.")

2025-12-19 22:02:17.717 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpz5dxxcmq.py capture_output=True code_len=2253
2025-12-19 22:02:17.746 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-19 22:02:17.750 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpwa4h1k7x.py capture_output=True code_len=2362
2025-12-19 22:02:17.772 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-19 22:02:17.774 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmps1nghv2w.py capture_output=True code_len=2262
2025-12-19 22:02:17.798 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-19 22:02:17.800 | DEBUG    | common.sandbox:execute_code:543 -

Executing test with input: 4 3
3 1
4 1
5 9
2 6
C0 Output: 1000000000000000000000000000000, C1 Output: 1000000000000000000000000000000, Expected: 15
Executing test with input: 4 1000000000
3 3
3 3
3 3
3 3
C0 Output: 0, C1 Output: 0, Expected: 0
Executing test with input: 4 1
1000000000 1000000000
1000000000 1000000000
1000000000 1000000000
1 1
C0 Output: 1000000000000000000000000000000, C1 Output: 1000000000000000000000000000000, Expected: 5999999994
Executing test with input: 15 128
748 169
586 329
972 529
432 519
408 587
138 249
656 114
632 299
984 755
404 772
155 506
832 854
353 465
387 374
567 385


2025-12-19 22:02:17.920 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0


C0 Output: 1000000000000000000000000000000, C1 Output: 1000000000000000000000000000000, Expected: 9460
The two code snippets are functionally equivalent on all public test cases of problem abc395_f.


In [350]:
public_test_input_outputs

[{'inputdata': '4 3\n3 1\n4 1\n5 9\n2 6',
  'output': '1000000000000000000000000000000'},
 {'inputdata': '4 1000000000\n3 3\n3 3\n3 3\n3 3', 'output': '0'},
 {'inputdata': '4 1\n1000000000 1000000000\n1000000000 1000000000\n1000000000 1000000000\n1 1',
  'output': '1000000000000000000000000000000'},
 {'inputdata': '15 128\n748 169\n586 329\n972 529\n432 519\n408 587\n138 249\n656 114\n632 299\n984 755\n404 772\n155 506\n832 854\n353 465\n387 374\n567 385',
  'output': '1000000000000000000000000000000'}]

# LLM CLient

In [351]:
import common
import importlib

importlib.reload(common.llm_client)

from common.llm_client import create_llm_client
from common.code_preprocessing.extraction import extract_code_block_from_response


llm_client = create_llm_client(provider='openai', model='gpt-5-mini', reasoning_effort='minimal')

2025-12-19 22:02:34.369 | INFO     | common.llm_client:create_llm_client:398 - Creating LLM client: provider=openai, model=gpt-5-mini
2025-12-19 22:02:34.369 | DEBUG    | common.llm_client:__init__:47 - Initialized LLM client: model=gpt-5-mini, max_tokens=1000000, limit_enabled=True
2025-12-19 22:02:34.395 | DEBUG    | common.llm_client:__init__:249 - Initialized OpenAIClient with model: gpt-5-mini, reasoning_effort: minimal
2025-12-19 22:02:34.395 | INFO     | common.llm_client:create_llm_client:428 - Successfully created OpenAIClient


# Solution explanations

In [352]:
EXPLAIN_PROMPT_TEMPLATE = """
<task>
- You are given a programming problem in <problem> and a code snippet in <code_snippet> that is believed to solve the problem.
- Your task is to explain in detail how the code snippet works to solve the problem.
- Provide a step-by-step explanation of the logic and approach used in the code snippet.
</task>

<problem>
{question_content}
</problem>

<code_snippet>
{code_snippet}
</code_snippet>
"""

In [353]:
C0_explanation = llm_client.generate(
    EXPLAIN_PROMPT_TEMPLATE.format(
        question_content=problem.question_content,
        code_snippet=C0_snippet
)
)

C1_explanation = llm_client.generate(
    EXPLAIN_PROMPT_TEMPLATE.format(
        question_content=problem.question_content,
        code_snippet=C1_snippet
)
)

print("C0 Explanation:")
print(C0_explanation)
print("\nC1 Explanation:")
print(C1_explanation)

2025-12-19 22:02:38.212 | DEBUG    | common.llm_client:generate:282 - OpenAIClient generating with model: gpt-5-mini, reasoning_effort: minimal
2025-12-19 22:03:46.673 | DEBUG    | common.llm_client:generate:302 - Generated 10033 characters
2025-12-19 22:03:46.674 | DEBUG    | common.llm_client:generate:282 - OpenAIClient generating with model: gpt-5-mini, reasoning_effort: minimal
2025-12-19 22:04:17.366 | DEBUG    | common.llm_client:generate:302 - Generated 7373 characters


C0 Explanation:
I'll explain the provided code snippet step by step: what it computes, the reasoning behind each step, and why it solves the problem (or where it departs from an exact solution idea).

High-level goal
- We must reduce tooth lengths (paying 1 yen per unit reduced on any single tooth) so that:
  1. There is an integer H with U_i + D_i = H for every i after reductions (i.e., for each position i the upper + lower equals the same H).
  2. The adjacent upper teeth differences satisfy |U_i - U_{i+1}| <= X for every adjacent pair after reductions.
- We can only decrease individual tooth lengths, so final upper values A_i satisfy 0 <= A_i <= original U_i, and final lower values B_i satisfy 0 <= B_i <= original D_i, and A_i + B_i = H.

Overview of the approach used in the code
1. Consider candidate values of H. If there's a final H, then for each position i the final upper tooth must be exactly T_i = H - D_i. That T_i must be between 0 and U_i inclusive; otherwise that H is impos

# DET Prompt

In [366]:
DET_PROMPT = """
<task>
- You are given a programming problem described in a <problem> block. 
- You are also given two Python code snippets in <solution variant='P'> and <solution variant='Q'> blocks that are intended to solve the same programming problem. 
- Additionally, you have access to explanations of how each code snippet works in the <explanation variant='P'> and <explanation variant='Q'> blocks.
- You are also given the current test inputs and the corresponding equivalent outputs they produce for both P and Q solutions in a <current_tests> block.
- The current tests fail to differentiate between the two code snippets as they produce the same outputs for both.
- Your task is to generate a new test input that can differentiate between the two code snippets.
- Consider the following guidelines while generating the test input:
    -- Analyze the problem statement to understand the requirements and constraints.
    -- In the problem, identify the input format, and input constraints, and any special conditions that need to be handled.
    -- Review the code solutions with the explanations provided to identify any differences in logic, approach, or edge case handling between the two.
    -- Think of edge cases, boundary conditions, very large input cases, and scenarios that might lead to different behaviors in the two implementations.
    -- Consider inputs that test the limits of the problem constraints or that exploit any identified weaknesses in either code snippet.
    -- New test input should be completely different from the existing test inputs provided in the <current_tests> block.
</task>

<problem>
{question_content}
</problem>


<solution variant='P'>
```python
{code_snippet_P}
```
</solution>

<explanation variant='P'>
{explanation_P}
</explanation>

<solution variant='Q'>
```python
{code_snippet_Q}
```
</solution>

<explanation variant='Q'>
{explanation_Q}
</explanation>

<current_tests>
{current_tests}
</current_tests>

<output_format>
Provide a single test input in the following Python dict format inside a code block:
```python
{{"inputdata": {{<your_generated_test_input>"}}}}
```

eg: for stdin test case:
```python
{{"inputdata": {{"input_str": "test input string"}}}}
```
for functional test case:
```python
{{"inputdata": {{"param1": value1, "param2": value2, ...}}}}
```

</output_format>
"""

In [367]:
import random
current_tests = public_test_input_outputs.copy()

for i in range(10):
    print(f"Generating differential test input, attempt {i+1}/10...")
    prompt = DET_PROMPT.format(
        question_content=problem.question_content,
        code_snippet_P=C0_snippet,
        explanation_P=C0_explanation,
        code_snippet_Q=C1_snippet,
        explanation_Q=C1_explanation,
        current_tests="\n".join(json.dumps(t) for t in random.sample(current_tests, min(len(current_tests), 20))),
    )
    response = llm_client.generate(prompt=prompt)

    test_input = extract_code_block_from_response(response)
    print("Generated differential test input:", test_input)

    c0_script = compose_lcb_output_script(c0_ind.snippet, test_input)
    c0_result = sandbox.execute_code(c0_script).output.strip()

    c1_script = compose_lcb_output_script(c1_ind.snippet, test_input)
    c1_result = sandbox.execute_code(c1_script).output.strip()


    if c0_result != c1_result:
        print(f"Differential test input found on attempt {i+1}!")
        print("Test Input:", test_input)
        print("C0 Output:", c0_result)
        print("C1 Output:", c1_result)
        break

    else:
        print(f"No discrepancy found on attempt {i+1}. Trying again...")
        current_tests.append({'input': json.loads(test_input)["inputdata"], 'output': c0_result})



2025-12-19 22:13:35.195 | DEBUG    | common.llm_client:generate:282 - OpenAIClient generating with model: gpt-5-mini, reasoning_effort: minimal


Generating differential test input, attempt 1/10...


2025-12-19 22:13:37.211 | DEBUG    | common.llm_client:generate:302 - Generated 49 characters
2025-12-19 22:13:37.216 | DEBUG    | common.code_preprocessing.composition:compose_lcb_output_script:305 - Error processing input data: {"inputdata": "3 1\n2 2\n1 3\n2 2"}


Generated differential test input: {"inputdata": "3 1\n2 2\n1 3\n2 2"}


CodeTransformationError: Error processing input data: 'inputdata' value must be a dict object

# Input Generation

In [368]:
INPUT_GENERATION_PROMPT = """
<task>
- You are given a programming problem described in a <problem> block. 
- You are also given two Python code snippets in <solution variant='P'> and <solution variant='Q'> blocks that are intended to solve the same programming problem. 
- Additionally, you have access to explanations of how each code snippet works in the <explanation variant='P'> and <explanation variant='Q'> blocks.
- You are also given the current test inputs and the corresponding equivalent outputs they produce for both P and Q solutions in a <current_tests> block.
- The current tests fail to differentiate between the two code snippets as they produce the same outputs for both.
- Your task is to write a python script that generates test inputs that can differentiate between the two code snippets.
- The script should define a function named `generate_test_inputs` that takes an integer parameter specifying the number of test inputs to generate and returns a list of dictionaries, each representing a test input.
- The script should be able to generate multiple diverse test inputs as much as required by the user in a single execution. 
    -- Eg: if the user requests 100 test inputs, the script should generate 100 test inputs in one go.
- You are allowed to use standard python libraries to implement the script.
- You are allowed to define helper functions within the script to aid in generating the test inputs.
- You are NOT allowed to manually hardcode any test inputs within the script.
</task>

<problem>
{question_content}
</problem>

<solution variant='P'>
```python
{code_snippet_P}
```
</solution>

<solution variant='Q'>
```python
{code_snippet_Q}
```
</solution>


<current_tests>
{current_tests}
</current_tests>

<output_format>
Provide only the python script inside a code block as follows:
```python
def generate_test_inputs(num_inputs: int) -> list[dict]:
    # Your code to generate test inputs
```
if input is via stdin, each dict should be of the form:
{{"input_str": "<generated_input_string>"}}
if input is functional, each dict should be of the form:
{{"param1": value1, "param2": value2, ...}}
</output_format>

"""

In [371]:
from common.code_preprocessing.transformation import remove_if_main_block
response = llm_client.generate(
    INPUT_GENERATION_PROMPT.format(
        question_content=problem.question_content,
        code_snippet_P=C0_snippet,
        code_snippet_Q=C1_snippet,
        current_tests="\n".join(json.dumps(t) for t in public_test_input_outputs),
)
)

generated_script = extract_code_block_from_response(response)
generated_script = remove_if_main_block(generated_script)
generated_script += """\n
print(generate_test_inputs(100))
"""

print(generated_script)

2025-12-19 22:14:13.811 | DEBUG    | common.llm_client:generate:282 - OpenAIClient generating with model: gpt-5-mini, reasoning_effort: minimal
2025-12-19 22:14:46.025 | DEBUG    | common.llm_client:generate:302 - Generated 4145 characters


import random
from typing import List, Dict

def _rand_pair(max_val):
    return (random.randint(1, max_val), random.randint(1, max_val))

def _build_case(N, X, U, D):
    parts = [f'{N} {X}']
    for u, d in zip(U, D):
        parts.append(f'{u} {d}')
    return '\n'.join(parts)

def _candidate_simple_diff(N, X):
    U = []
    D = []
    for i in range(N):
        if i % 2 == 0:
            u = random.randint(1, 10 ** 9 // 2)
        else:
            u = random.randint(1, 10 ** 9)
        d = random.randint(1, 10 ** 9 // 2)
        U.append(u)
        D.append(d)
    return (U, D)

def _candidate_edge_cases(N, X):
    U = []
    D = []
    baseS = random.randint(1, 10 ** 9)
    mode = random.choice(['sameS', 'descendingS', 'increasingU', 'random'])
    for i in range(N):
        if mode == 'sameS':
            u = random.randint(1, baseS - 1) if baseS > 2 else 1
            d = baseS - u
            if d <= 0:
                d = 1
                u = baseS - d
        elif mode == 

In [373]:
from common.code_preprocessing.analysis import validate_code_syntax
import ast

output = sandbox.execute_code(generated_script).output
if not validate_code_syntax(output):
    raise ValueError("Generated script has syntax errors.")


test_inputs: list[dict] = ast.literal_eval(output)
print("Number of generated test inputs:", len(test_inputs))
print(test_inputs)

2025-12-19 22:15:04.295 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpq2pa588h.py capture_output=True code_len=3502
2025-12-19 22:15:05.209 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0


Number of generated test inputs: 100
[{'input_str': '160 1\n230035496 680\n439 304734326\n319039808 826232404\n605060171 417\n435 781233735\n788612892 354255776\n794753691 852\n956 273556408\n819131961 981386384\n554306797 651\n445 428078949\n728667316 106161490\n471450379 794\n118 468550657\n164428365 912669687\n366587264 440\n814 174914767\n142645122 572008978\n814849676 205\n791 504570618\n592079954 512476227\n568751072 873\n192 292455228\n778194424 886399432\n411135369 791\n32 173805765\n544975734 278306681\n344419607 412\n683 926754711\n215764372 112267997\n399431750 346\n6 644812665\n48143240 963160624\n781877086 70\n110 355879939\n357347866 882808772\n324577844 953\n809 753708908\n110953814 595074032\n240620333 857\n335 229002540\n281138256 123618035\n384414094 889\n213 986775338\n109727439 422410495\n596090666 932\n706 806956775\n16216304 346396403\n801595509 450\n835 658596352\n415774457 495134430\n637511845 314\n555 273005153\n286503997 344162392\n841260298 485\n575 615265877

In [374]:
for idx, ti in enumerate(test_inputs):
    ti_formatted = {"inputdata": ti}
    c0_script = compose_lcb_output_script(c0_ind.snippet, str(ti_formatted))
    c0_result = sandbox.execute_code(c0_script).output.strip()
    c1_script = compose_lcb_output_script(c1_ind.snippet, str(ti_formatted))
    c1_result = sandbox.execute_code(c1_script).output.strip()
    if c0_result != c1_result:
        print(f"Test Input {idx+1}:\n")
        print("Discrepancy found!")
        print("Test Input:", ti)
        print("C0 Output:", c0_result)
        print("C1 Output:", c1_result)
        break

2025-12-19 22:15:17.065 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpsyjqqfx3.py capture_output=True code_len=4934
2025-12-19 22:15:17.977 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-19 22:15:17.981 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpbe3kt0pw.py capture_output=True code_len=5043
2025-12-19 22:15:18.915 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-19 22:15:18.919 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmp3jpaelvd.py capture_output=True code_len=2649
2025-12-19 22:15:19.839 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-19 22:15:19.843 | DEBUG    | common.sandbox:execute_code:543 -

Test Input 4:

Discrepancy found!
Test Input: {'input_str': '192 740151175\n260802626 609551516\n680239675 190114467\n7711671 862642471\n250739628 619614514\n432991814 437362328\n23265675 847088467\n537623740 332730402\n89696782 780657360\n163012370 707341772\n52214141 818140001\n868879575 1474567\n15407512 854946630\n622902963 247451179\n208973443 661380699\n433924673 436429469\n267344717 603009425\n89964845 780389297\n844506327 25847815\n708414898 161939244\n666763546 203590596\n703974666 166379476\n796036368 74317774\n313529686 556824456\n578267085 292087057\n426804821 443549321\n704914186 165439956\n517127985 353226157\n618909037 251445105\n3374746 866979396\n420188690 450165452\n509769210 360584932\n574822565 295531577\n315746403 554607739\n74723078 795631064\n600350604 270003538\n55446705 814907437\n779944566 90409576\n320973567 549380575\n685596274 184757868\n857373911 12980231\n722765070 147589072\n255169350 615184792\n382924942 487429200\n773337450 97016692\n182801456 68755268